# YOLOv8 Computer Apparatus Detection Model
This notebook trains a YOLOv8 model on the electronics dataset (mouse, keyboard, laptop) and uploads the best model to HuggingFace Hub.

## 1. Install Dependencies

In [2]:
!pip install ultralytics kagglehub huggingface-hub opencv-python pyyaml pillow -q

In [3]:
import os
import shutil
import json
from pathlib import Path
from xml.etree import ElementTree as ET
import kagglehub
from huggingface_hub import HfApi, login
from ultralytics import YOLO
import cv2
from PIL import Image
import yaml

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
import os
import getpass
from google.colab import files

print("\n" + "="*60)
print("KAGGLE AUTHENTICATION")
print("="*60)
print("\nChoose your authentication method:")
print("  1. Paste Kaggle API Token (RECOMMENDED)")
print("  2. Upload kaggle.json file")
print("\nYour token will NOT be saved or displayed.")

choice = input("\nEnter choice (1 or 2): ").strip()

if choice == "1":
    print("\n📝 Enter your Kaggle API token")
    print("   Get it from: https://www.kaggle.com/settings/account")
    print("   Click 'Create New API Token' and copy the token value\n")

    kaggle_token = "xxxxx"

    if not kaggle_token:
        raise ValueError("Token cannot be empty!")

    os.environ['KAGGLE_API_TOKEN'] = kaggle_token

    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)

    print("\nKaggle credentials configured using API token!")
    print("   Token is stored in memory for this session only.")

elif choice == "2":

    uploaded = files.upload()

    if 'kaggle.json' not in uploaded:
        raise FileNotFoundError("kaggle.json file not found!")

    # Setup Kaggle
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    !mv kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json

    print("\nKaggle credentials configured using kaggle.json!")
else:
    raise ValueError("Invalid choice. Please enter 1 or 2.")

print("\n" + "="*60)


KAGGLE AUTHENTICATION

Choose your authentication method:
  1. Paste Kaggle API Token (RECOMMENDED)
  2. Upload kaggle.json file

Your token will NOT be saved or displayed.

Enter choice (1 or 2): 1

📝 Enter your Kaggle API token
   Get it from: https://www.kaggle.com/settings/account
   Click 'Create New API Token' and copy the token value


Kaggle credentials configured using API token!
   Token is stored in memory for this session only.



In [22]:
import os
import random
import shutil
from pathlib import Path
import xml.etree.ElementTree as ET
import kagglehub

# ---------------- CONFIG ----------------
DATASET_PATH = kagglehub.dataset_download("dataclusterlabs/electronics-mouse-keyboard-image-dataset")
ANNOTATIONS_DIR = os.path.join(DATASET_PATH, "annotations", "annotations")
IMAGES_DIR = os.path.join(DATASET_PATH, "samples_for_clients", "samples_for_clients")

YOLO_ROOT = "/content/yolo_dataset"
TRAIN_RATIO = 0.8

# Replace this with your actual classes in XML
CLASSES = {
    "mouse": 0,
    "keyboard":1,
    "monitor":2,
}

# ----------------------------------------

def convert_voc_to_yolo(xml_path, txt_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    size = root.find("size")
    img_w = float(size.find("width").text)
    img_h = float(size.find("height").text)

    lines = []

    for obj in root.findall("object"):
        cls_name = obj.find("name").text.strip()

        if cls_name not in CLASSES:
            continue

        cls_id = CLASSES[cls_name]
        bnd = obj.find("bndbox")

        xmin = float(bnd.find("xmin").text)
        ymin = float(bnd.find("ymin").text)
        xmax = float(bnd.find("xmax").text)
        ymax = float(bnd.find("ymax").text)

        x_center = ((xmin + xmax) / 2) / img_w
        y_center = ((ymin + ymax) / 2) / img_h
        width = (xmax - xmin) / img_w
        height = (ymax - ymin) / img_h

        lines.append(
            f"{cls_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}"
        )

    with open(txt_path, "w") as f:
        f.write("\n".join(lines))


# ---------------- CREATE DIRS ----------------
for split in ["train", "val"]:
    os.makedirs(f"{YOLO_ROOT}/images/{split}", exist_ok=True)
    os.makedirs(f"{YOLO_ROOT}/labels/{split}", exist_ok=True)

# ---------------- FIND FILES ----------------
images = sorted([f for f in os.listdir(IMAGES_DIR) if f.lower().endswith(".jpg")])
random.shuffle(images)

split_idx = int(len(images) * TRAIN_RATIO)
train_imgs = images[:split_idx]
val_imgs = images[split_idx:]

def process_split(image_list, split):
    for img_name in image_list:
        base = Path(img_name).stem
        img_src = os.path.join(IMAGES_DIR, img_name)
        xml_src = os.path.join(ANNOTATIONS_DIR, f"{base}.xml")

        if not os.path.exists(xml_src):
            print(f"⚠️ Missing XML for {img_name}")
            continue

        img_dst = f"{YOLO_ROOT}/images/{split}/{img_name}"
        txt_dst = f"{YOLO_ROOT}/labels/{split}/{base}.txt"

        shutil.copy(img_src, img_dst)
        convert_voc_to_yolo(xml_src, txt_dst)

process_split(train_imgs, "train")
process_split(val_imgs, "val")

print("✅ VOC → YOLO conversion complete")
print(f"Train images: {len(train_imgs)}")
print(f"Val images: {len(val_imgs)}")


Using Colab cache for faster access to the 'electronics-mouse-keyboard-image-dataset' dataset.
✅ VOC → YOLO conversion complete
Train images: 80
Val images: 20


In [23]:
import os
import yaml

# ---------------- PRINT DIRECTORY TREE ----------------
def print_tree(path, indent=0, max_depth=4):
    if indent > max_depth:
        return
    items = sorted(os.listdir(path))
    for i, item in enumerate(items):
        full = os.path.join(path, item)
        branch = "└── " if i == len(items) - 1 else "├── "
        print("  " * indent + branch + item)
        if os.path.isdir(full):
            print_tree(full, indent + 1, max_depth)

print_tree("/content/yolo_dataset")

# ---------------- CREATE YAML FOR YOLO ----------------
YOLO_ROOT = "/content/yolo_dataset"

# Automatically detect classes from labels
labels_dir = os.path.join(YOLO_ROOT, "labels", "train")
classes_set = set()
for txt_file in os.listdir(labels_dir):
    if txt_file.endswith(".txt"):
        with open(os.path.join(labels_dir, txt_file)) as f:
            for line in f:
                cls_id = int(line.split()[0])
                classes_set.add(cls_id)

# Sort classes by ID
classes_sorted = sorted(list(classes_set))

# Map class IDs to names (must match previous CLASSES dict)
CLASSES = {
    "mouse": 0,
    "keyboard":1,
    "monitor":2,
}
names = [name for name, idx in sorted(CLASSES.items(), key=lambda x: x[1])]

yaml_data = {
    "path": YOLO_ROOT,
    "train": "images/train",
    "val": "images/val",
    "nc": len(names),
    "names": names
}

yaml_path = "/content/computerApparatus.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(yaml_data, f, sort_keys=False)

print(f"✅ YOLO dataset YAML created at: {yaml_path}")


├── images
  ├── train
    ├── 20201230_15_02_32_000_M9gVSRUsnJYxozuIsk4ReQjamuJ2_F_3264_2448.jpg.jpg
    ├── 20201230_15_08_20_000_M9gVSRUsnJYxozuIsk4ReQjamuJ2_F_3264_2448.jpg.jpg
    ├── 20201230_15_21_44_000_M9gVSRUsnJYxozuIsk4ReQjamuJ2_T_4160_3120.jpg.jpg
    ├── 20201230_15_22_20_000_M9gVSRUsnJYxozuIsk4ReQjamuJ2_T_4160_3120.jpg.jpg
    ├── 20201230_17_02_33_000_MHvmWMWftvMzzFmedX6ol5Q5I5t2_T_3000_4000.jpg.jpg
    ├── 20201231_11_10_08_000_MHvmWMWftvMzzFmedX6ol5Q5I5t2_F_3264_2448.jpg.jpg
    ├── 20201231_11_14_42_000_MHvmWMWftvMzzFmedX6ol5Q5I5t2_F_3264_2448.jpg.jpg
    ├── 20201231_11_18_07_000_MHvmWMWftvMzzFmedX6ol5Q5I5t2_F_3264_2448.jpg.jpg
    ├── 20201231_14_57_25_000_E6PxFNY1Vsb8ebbISmYzvCKKshS2_F_3264_2448.jpg.jpg
    ├── 20201231_14_58_03_000_E6PxFNY1Vsb8ebbISmYzvCKKshS2_F_3264_2448.jpg.jpg
    ├── 20201231_18_31_31_000_r8FuglH9p9cGrfnIFjfWEW9V3EG3_T_1968_4144.jpg.jpg
    ├── 20201231_18_32_17_000_r8FuglH9p9cGrfnIFjfWEW9V3EG3_T_1968_4144.jpg.jpg
    ├── 20201231_22_20_10_000

In [ ]:
from huggingface_hub import login
import getpass

print("\n" + "="*60)
print("🤗 HUGGING FACE AUTHENTICATION")

hf_token ="yyyyyy"

# Login to Hugging Face
try:
    login(token=hf_token)
    print("\n  Logged in to Hugging Face successfully!")
except Exception as e:
    print(f"\n  Authentication failed: {str(e)}")
    raise



from huggingface_hub import hf_hub_download
from ultralytics import YOLO

REPO_ID = "IndUSV/Yolov8_SE_3"
FILENAME = "Yolov8_SE_2.pt"   # ⚠️ must be a YOLO .pt file

model_path = hf_hub_download(
    repo_id=REPO_ID,
    filename=FILENAME
)

print("✅ Downloaded model to:", model_path)




🤗 HUGGING FACE AUTHENTICATION

  Logged in to Hugging Face successfully!
✅ Downloaded model to: /root/.cache/huggingface/hub/models--IndUSV--Yolov8_SE_3/snapshots/6ed392d67225b919174edc84f79f7a7d16f3db38/Yolov8_SE_2.pt


In [26]:
from ultralytics import YOLO

model = YOLO(model_path)

print("\n" + "="*60)
print("STARTING MODEL TRAINING")
print("="*60 + "\n")

epochs = 50
imgsz = 640
batch_size = 16
patience = 10

print("Training configuration:")
print(f"  • Model: YOLOv8n")
print(f"  • Epochs: {epochs}")
print(f"  • Image size: {imgsz}")
print(f"  • Batch size: {batch_size}")
print(f"  • Early stopping patience: {patience}")
print(f"  • Dataset YAML: {yaml_path}\n")

results = model.train(
    data=yaml_path,
    epochs=epochs,
    imgsz=imgsz,
    batch=batch_size,
    patience=patience,
    device="cpu",
    project="computerApparatus_detector",
    name="yolov8n_v1",
    exist_ok=False,
    save=True,
    save_period=5,
    plots=True,
    verbose=True
)

print("\n" + "="*60)
print("TRAINING COMPLETED")
print("="*60)



STARTING MODEL TRAINING

Training configuration:
  • Model: YOLOv8n
  • Epochs: 50
  • Image size: 640
  • Batch size: 16
  • Early stopping patience: 10
  • Dataset YAML: /content/computerApparatus.yaml

Ultralytics 8.4.12 🚀 Python-3.12.12 torch-2.9.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/computerApparatus.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300

## 11. Verify Best Model

In [31]:
from ultralytics import YOLO

# Load trained model
trained_model = YOLO("/content/runs/detect/computerApparatus_detector/yolov8n_v12/weights/best.pt")

# Run validation
results = trained_model.val(
    data=yaml_path,  # your dataset YAML
    imgsz=640,
    batch=16,
    device="cpu",    # change to "0" if using GPU
    verbose=True
)

# Access metrics using results.stats or results.mean_results()
precision, recall, mAP50, mAP50_95 = results.mean_results()

print("\n✅ Validation Results Summary:")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"mAP50: {mAP50:.4f}")
print(f"mAP50-95: {mAP50_95:.4f}")


Ultralytics 8.4.12 🚀 Python-3.12.12 torch-2.9.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
Model summary (fused): 73 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2924.5±702.8 MB/s, size: 2608.4 KB)
val: Scanning /content/yolo_dataset/labels/val.cache... 38 images, 18 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 38/38 10.0Mit/s 0.0s
val: /content/yolo_dataset/images/val/20210106_11_59_43_000_2ma82BCxRqPzpTTB8Z98xDtLUry2_F_3000_4000.jpg.jpg: corrupt JPEG restored and saved
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 4.1s/it 12.3s
                   all         38         30      0.654      0.283      0.231      0.173
                 mouse          9          9       0.61       0.35      0.394      0.339
              keyboard         16         16      0.351        0.5      0.283      0.175
               monitor          4          5          1          0     0.0151    0.

## 12. Login to HuggingFace Hub

In [ ]:
from huggingface_hub import login

hf_token = "yyyyy"
login(token=hf_token)


## 13. Create Repository and Upload Model

In [36]:
from huggingface_hub import HfApi
from pathlib import Path

# =========================
# CONFIGURATION
# =========================

REPO_ID = "IndUSV/computerApparatus-detector"

BEST_MODEL_PATH = Path(
    "/content/runs/detect/computerApparatus_detector/yolov8n_v12/weights/best.pt"
)

# =========================
# INIT HF API
# =========================

api = HfApi()

print("\n" + "=" * 60)
print(" PUSHING TRAINED MODEL TO HUGGING FACE")
print("=" * 60)
print(f"\nTarget repository: {REPO_ID}\n")

# =========================
# CREATE / VERIFY REPO
# =========================

api.create_repo(
    repo_id=REPO_ID,
    repo_type="model",
    exist_ok=True,
    private=False
)

print("Repository ready")

# =========================
# CHECK MODEL
# =========================

if not BEST_MODEL_PATH.exists():
    raise FileNotFoundError(f"Model not found: {BEST_MODEL_PATH}")

print(f"Model found: {BEST_MODEL_PATH}")
print(f"Size: {BEST_MODEL_PATH.stat().st_size / (1024*1024):.2f} MB\n")

# =========================
# UPLOAD MODEL (CORRECT NAME)
# =========================

print("Uploading model weights...")

api.upload_file(
    path_or_fileobj=str(BEST_MODEL_PATH),
    path_in_repo="yolov8n_best.pt",  # name inside HF repo
    repo_id=REPO_ID,
    repo_type="model"
)

print("✅ Model uploaded successfully!")



 PUSHING TRAINED MODEL TO HUGGING FACE

Target repository: IndUSV/computerApparatus-detector

Repository ready
Model found: /content/runs/detect/computerApparatus_detector/yolov8n_v12/weights/best.pt
Size: 5.94 MB

Uploading model weights...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...lov8n_v12/weights/best.pt:  86%|########5 | 5.33MB / 6.23MB            

✅ Model uploaded successfully!
